# GAT Attention 비교: gnn_266_phase3 vs gnn_266_phase3_noipc

각 모델에서 레이어별·엣지 타입별 평균 GAT attention weight를 추출하고 비교한다.

- `phase3_simplecat_ckpt.pt` : IPC 엣지 포함 (8개 엣지 타입)
- `phase3_noipc_ckpt.pt`     : IPC 엣지 제외 (6개 엣지 타입)

> **alpha 해석**: PyG `GATConv(return_attention_weights=True)`가 반환하는 alpha는  
> 각 도착 노드에 대한 incoming edge softmax 가중치 (shape: `[E, H]`).  
> 여기서는 **전체 엣지 × 헤드 평균**을 '해당 엣지 타입의 평균 attention'으로 사용.

In [ ]:
# Cell 1 — Imports
import re, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, GATConv

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CKPT_DIR = Path('..').resolve()          # src/data_266/
DATA_DIR = Path('../../..').resolve() / 'data' / 'samples'

print('device :', device)
print('ckpt_dir:', CKPT_DIR)
print('data_dir:', DATA_DIR)

In [ ]:
# Cell 2 — 공통 데이터 로드 (phase3_simplecat_ckpt.pt 기준)
# noipc 체크포인트도 df_cit/df_ipc/df_asgn 등은 simplecat_ckpt에서 가져왔으므로 동일

ckpt_ipc   = torch.load(CKPT_DIR / 'phase3_simplecat_ckpt.pt',  weights_only=False)
ckpt_noipc = torch.load(CKPT_DIR / 'phase3_noipc_ckpt.pt',      weights_only=False)

# 공통
pat_list   = ckpt_ipc['pat_list']
pat_to_idx = ckpt_ipc['pat_to_idx']
co_list    = ckpt_ipc['co_list']
co_to_idx  = ckpt_ipc['co_to_idx']
ipc_list   = ckpt_ipc['ipc_list']
ipc_to_idx = ckpt_ipc['ipc_to_idx']

pat_emb    = ckpt_ipc['pat_emb']     # (N_pat, 1024)
pat_struct = ckpt_ipc['pat_struct']  # (N_pat, 3)
co_feat    = ckpt_ipc['co_feat']     # (N_co, 1024)

df_cit  = ckpt_ipc['df_cit']
df_ipc  = ckpt_ipc['df_ipc']
df_asgn = ckpt_ipc['df_asgn']

print(f'Patents  : {len(pat_list):,}')
print(f'Companies: {len(co_list):,}')
print(f'IPC codes: {len(ipc_list):,}')
print(f'df_cit   : {len(df_cit):,} edges')
print(f'df_ipc   : {len(df_ipc):,} edges')
print(f'df_asgn  : {len(df_asgn):,} edges')

In [ ]:
# Cell 3 — Train sued edges 재구성 (both_side_patents.csv + train_cases)
_SUFFIX = re.compile(
    r',?\s*('
    r'incorporated|corporation|company|limited liability company'
    r'|limited liability partnership|limited partnership|limited'
    r'|inc|corp|co|ltd|llc|llp|lp|plc|gmbh|b\.?v|n\.?v|s\.?a|a\.?g|s\.?r\.?l'
    r'|pvt|pte|pty|ab|as|a\/s|oy|oyj|spa|srl|sarl|sas|bv|nv'
    r'|an? (?:delaware|california|new\s+york|new\s+jersey|texas|illinois|florida'
    r'|washington|ohio|michigan|minnesota|pennsylvania|nevada|maryland|virginia'
    r'|massachusetts|colorado|georgia|indiana|wisconsin|arizona|connecticut'
    r'|netherlands|korean?|japanese|german|french|canadian|british|swedish|swiss)'
    r'\s+(?:corporation|limited\s+liability(?:\s+company)?|limited\s+partnership|limited|company)'
    r'|doing business as|terminated.*'
    r')[,.]?\s*$',
    re.IGNORECASE
)
_PREFIX = re.compile(r'^(the|a|an)\s+', re.IGNORECASE)

def deep_normalize(name):
    s = str(name).strip().replace('&', 'and')
    s = re.sub(r'[^\w\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip().upper()
    for _ in range(6):
        new = _SUFFIX.sub('', s, count=1).strip()
        if new == s: break
        s = new
    return _PREFIX.sub('', s).strip()

_mm_path = DATA_DIR / 'company_merge_map.csv'
MERGE_MAP = {}
if _mm_path.exists():
    _mm = pd.read_csv(_mm_path)
    MERGE_MAP = dict(zip(_mm['from'], _mm['to']))

def canonical(name):
    n = deep_normalize(name)
    return MERGE_MAP.get(n, n)

def parse_pipe(s):
    if pd.isna(s): return []
    return [x.strip() for x in str(s).split('|') if x.strip()]

raw = pd.read_csv(DATA_DIR / 'both_side_patents.csv')
rows = []
for _, r in raw.iterrows():
    for pat in parse_pipe(r['P_patents']):
        rows.append({'case_row_id': r['case_row_id'], 'patent_id': pat, 'side': 'plaintiff'})
    for pat in parse_pipe(r['D_patents']):
        rows.append({'case_row_id': r['case_row_id'], 'patent_id': pat, 'side': 'defendant'})
df_pats = pd.DataFrame(rows)

# sued pairs + case 매핑
sued_src, sued_dst, sued_case = [], [], []
for case_id, grp in df_pats.groupby('case_row_id'):
    p_pats = grp[grp['side'] == 'plaintiff']['patent_id'].tolist()
    d_pats = grp[grp['side'] == 'defendant']['patent_id'].tolist()
    for pp in p_pats:
        for dp in d_pats:
            if pp in pat_to_idx and dp in pat_to_idx:
                sued_src.append(pat_to_idx[pp])
                sued_dst.append(pat_to_idx[dp])
                sued_case.append(case_id)

sued_edges = torch.tensor([sued_src, sued_dst], dtype=torch.long)

# train_cases는 체크포인트에서 복원
train_cases = set(ckpt_ipc['train_cases'])
tr_mask     = np.array([c in train_cases for c in sued_case])
train_edges = sued_edges[:, tr_mask]

print(f'Total sued edges : {sued_edges.shape[1]}')
print(f'Train sued edges : {train_edges.shape[1]}')

In [ ]:
# Cell 4 — HeteroData 구성 (with IPC + without IPC)

def make_edge(srcs, dsts):
    return torch.tensor([srcs, dsts], dtype=torch.long)

# ── 공통 엣지 ────────────────────────────────────────────────
cs, cd = [], []
for _, r in df_cit.iterrows():
    if r['patent_id'] in pat_to_idx and r['citation_patent_id'] in pat_to_idx:
        cs.append(pat_to_idx[r['patent_id']])
        cd.append(pat_to_idx[r['citation_patent_id']])
cit_ei  = make_edge(cs, cd)

co_s, p_d = [], []
for _, r in df_asgn.iterrows():
    if r['org_norm'] in co_to_idx and r['patent_id'] in pat_to_idx:
        co_s.append(co_to_idx[r['org_norm']])
        p_d.append(pat_to_idx[r['patent_id']])
owns_ei = make_edge(co_s, p_d)

# ── IPC 엣지 ────────────────────────────────────────────────
ps, ii = [], []
for _, r in df_ipc.iterrows():
    if r['patent_id'] in pat_to_idx and r['ipc_code'] in ipc_to_idx:
        ps.append(pat_to_idx[r['patent_id']])
        ii.append(ipc_to_idx[r['ipc_code']])
ipc_ei  = make_edge(ps, ii)

# ── data_ipc (phase3 — IPC 포함) ────────────────────────────
data_ipc = HeteroData()
data_ipc['patent'].struct  = torch.tensor(pat_struct, dtype=torch.float32)
data_ipc['patent'].x       = torch.tensor(pat_emb,    dtype=torch.float32)
data_ipc['company'].x      = torch.tensor(co_feat,    dtype=torch.float32)
data_ipc['ipc'].num_nodes  = len(ipc_list)

data_ipc['patent',  'cites',        'patent'].edge_index = cit_ei
data_ipc['patent',  'cited_by',     'patent'].edge_index = make_edge(cd, cs)
data_ipc['patent',  'has_ipc',      'ipc'].edge_index    = ipc_ei
data_ipc['ipc',     'rev_has_ipc',  'patent'].edge_index = make_edge(ii, ps)
data_ipc['company', 'owns',         'patent'].edge_index = owns_ei
data_ipc['patent',  'owned_by',     'company'].edge_index = make_edge(p_d, co_s)
data_ipc['patent',  'sued',         'patent'].edge_index = train_edges
data_ipc['patent',  'sued_by',      'patent'].edge_index = torch.stack([train_edges[1], train_edges[0]])

# ── data_noipc (phase3_noipc — IPC 제외) ────────────────────
data_noipc = HeteroData()
data_noipc['patent'].struct  = torch.tensor(pat_struct, dtype=torch.float32)
data_noipc['patent'].x       = torch.tensor(pat_emb,    dtype=torch.float32)
data_noipc['company'].x      = torch.tensor(co_feat,    dtype=torch.float32)

data_noipc['patent',  'cites',    'patent'].edge_index  = cit_ei
data_noipc['patent',  'cited_by', 'patent'].edge_index  = make_edge(cd, cs)
data_noipc['company', 'owns',     'patent'].edge_index  = owns_ei
data_noipc['patent',  'owned_by', 'company'].edge_index = make_edge(p_d, co_s)
data_noipc['patent',  'sued',     'patent'].edge_index  = train_edges
data_noipc['patent',  'sued_by',  'patent'].edge_index  = torch.stack([train_edges[1], train_edges[0]])

print('data_ipc:')
for et, ei in data_ipc.edge_index_dict.items():
    print(f'  {str(et):<45} {ei.shape[1]:>5} edges')
print()
print('data_noipc:')
for et, ei in data_noipc.edge_index_dict.items():
    print(f'  {str(et):<45} {ei.shape[1]:>5} edges')

In [ ]:
# Cell 5 — 모델 정의

HID_DIM   = 128
OUT_DIM   = 64
FAISS_DIM = 1024
STRUCT_DIM = 3
DIM_TOTAL  = STRUCT_DIM + FAISS_DIM  # 1027

_CONV_ACTS = [F.leaky_relu, F.leaky_relu, torch.tanh]

EDGE_TYPES_IPC = [
    ('patent',  'cites',        'patent'),
    ('patent',  'cited_by',     'patent'),
    ('patent',  'has_ipc',      'ipc'),
    ('ipc',     'rev_has_ipc',  'patent'),
    ('company', 'owns',         'patent'),
    ('patent',  'owned_by',     'company'),
    ('patent',  'sued',         'patent'),
    ('patent',  'sued_by',      'patent'),
]

EDGE_TYPES_NOIPC = [
    ('patent',  'cites',        'patent'),
    ('patent',  'cited_by',     'patent'),
    ('company', 'owns',         'patent'),
    ('patent',  'owned_by',     'company'),
    ('patent',  'sued',         'patent'),
    ('patent',  'sued_by',      'patent'),
]


class SimpleLinearProj(nn.Module):
    def __init__(self, dim_total, hid, dropout=0.1):
        super().__init__()
        self.proj = nn.Linear(dim_total, hid)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(F.leaky_relu(self.proj(x)))


class Phase3Model(nn.Module):
    """gnn_266_phase3 — IPC 포함"""
    def __init__(self, n_ipc, hid=HID_DIM, out=OUT_DIM, drop=0.2):
        super().__init__()
        self.hid = hid; self.out = out
        self.pat_proj = SimpleLinearProj(DIM_TOTAL, hid, dropout=0.1)
        self.co_proj  = nn.Linear(FAISS_DIM, hid)
        self.ipc_emb  = nn.Embedding(n_ipc, hid)
        layer_cfg = [(hid//8,8,0.3),(hid//4,4,drop),(hid//4,4,drop),(out//2,2,drop)]
        self.convs = nn.ModuleList([
            HeteroConv({et: GATConv(hid,hd,heads=nh,dropout=ad,concat=True,add_self_loops=False)
                        for et in EDGE_TYPES_IPC}, aggr='mean')
            for hd,nh,ad in layer_cfg
        ])
        self.bns = nn.ModuleList([
            nn.ModuleDict({nt: nn.BatchNorm1d(hid) for nt in ['patent','company','ipc']})
            for _ in range(3)
        ])
        self.drop = nn.Dropout(drop)

    def forward(self, d):
        x_pat = torch.cat([d['patent'].struct, d['patent'].x], dim=1)
        x = {'patent': self.pat_proj(x_pat),
             'company': F.leaky_relu(self.co_proj(d['company'].x)),
             'ipc': self.ipc_emb.weight}
        for conv, bn, act in zip(self.convs[:3], self.bns, _CONV_ACTS):
            x_new = conv(x, d.edge_index_dict)
            x = {k: self.drop(act(bn[k](v.view(-1, self.hid)))) if k in bn
                 else self.drop(act(v)) for k, v in x_new.items()}
        x_out = self.convs[3](x, d.edge_index_dict)
        return {k: v.view(-1, self.out) for k, v in x_out.items()}


class Phase3NoIPCModel(nn.Module):
    """gnn_266_phase3_noipc — IPC 제외"""
    def __init__(self, hid=HID_DIM, out=OUT_DIM, drop=0.2):
        super().__init__()
        self.hid = hid; self.out = out
        self.pat_proj = SimpleLinearProj(DIM_TOTAL, hid, dropout=0.1)
        self.co_proj  = nn.Linear(FAISS_DIM, hid)
        layer_cfg = [(hid//8,8,0.3),(hid//4,4,drop),(hid//4,4,drop),(out//2,2,drop)]
        self.convs = nn.ModuleList([
            HeteroConv({et: GATConv(hid,hd,heads=nh,dropout=ad,concat=True,add_self_loops=False)
                        for et in EDGE_TYPES_NOIPC}, aggr='mean')
            for hd,nh,ad in layer_cfg
        ])
        self.bns = nn.ModuleList([
            nn.ModuleDict({nt: nn.BatchNorm1d(hid) for nt in ['patent','company']})
            for _ in range(3)
        ])
        self.drop = nn.Dropout(drop)

    def forward(self, d):
        x_pat = torch.cat([d['patent'].struct, d['patent'].x], dim=1)
        x = {'patent': self.pat_proj(x_pat),
             'company': F.leaky_relu(self.co_proj(d['company'].x))}
        for conv, bn, act in zip(self.convs[:3], self.bns, _CONV_ACTS):
            x_new = conv(x, d.edge_index_dict)
            x = {k: self.drop(act(bn[k](v.view(-1, self.hid)))) if k in bn
                 else self.drop(act(v)) for k, v in x_new.items()}
        x_out = self.convs[3](x, d.edge_index_dict)
        return {k: v.view(-1, self.out) for k, v in x_out.items()}


print('Phase3Model    (IPC)   defined')
print('Phase3NoIPCModel (no IPC) defined')

In [ ]:
# Cell 6 — 모델 가중치 로드

model_ipc = Phase3Model(n_ipc=len(ipc_list)).to(device)
model_ipc.load_state_dict(ckpt_ipc['model_state'])
model_ipc.eval()

model_noipc = Phase3NoIPCModel().to(device)
model_noipc.load_state_dict(ckpt_noipc['model_state'])
model_noipc.eval()

data_ipc_d   = data_ipc.to(device)
data_noipc_d = data_noipc.to(device)

print('phase3      (IPC):   best epoch', ckpt_ipc['best_epoch'],
      '  test AUROC', ckpt_ipc['test_result']['auroc'])
print('phase3_noipc:        best epoch', ckpt_noipc['best_epoch'],
      '  test AUROC', ckpt_noipc['test_result']['auroc'])

In [ ]:
# Cell 7 — GAT Attention 추출 함수 (수정판)
#
# [주의] mean(alpha) = N_dst / E_total  (softmax 수학적 성질)
#   → 모델 가중치와 무관하며 그래프 구조만 반영
#   → 두 모델에서 같게 나오는 것은 버그가 아님
#
# [올바른 지표] 노드별 attention 집중도:
#   - max_alpha : 각 dst 노드의 최대 attention (head 평균) → 집중도
#   - entropy   : 각 dst 노드의 attention entropy → 낮을수록 선택적
#   - gini      : 지니계수 → 높을수록 불균등 집중
#   노드별로 계산 후 전체 노드에 대해 평균

import math

@torch.no_grad()
def extract_attention(model, hetero_data, has_ipc=False):
    """
    Returns:
        layer_results: list of dict  (length=4)
            key   : (src_type, rel_type, dst_type)
            value : {
                'mean_alpha'  : mean(alpha) = N_dst/E  (구조적, 참고용)
                'node_max'    : mean over dst nodes of per-node max alpha (집중도)
                'node_entropy': mean over dst nodes of per-node entropy (낮을수록 선택적)
                'node_gini'   : mean over dst nodes of gini coefficient
                'n_edges': int, 'n_dst': int, 'n_heads': int
            }
    """
    model.eval()
    d = hetero_data

    x_pat = torch.cat([d['patent'].struct, d['patent'].x], dim=1)
    x = {'patent':  model.pat_proj(x_pat),
         'company': F.leaky_relu(model.co_proj(d['company'].x))}
    if has_ipc:
        x['ipc'] = model.ipc_emb.weight

    layer_results = []

    for layer_idx in range(4):
        hconv = model.convs[layer_idx]
        layer_attn = {}

        for et_key, gat_conv in hconv.convs.items():
            if isinstance(et_key, tuple):
                src_t, rel_t, dst_t = et_key
            else:
                parts = et_key.split('__')
                src_t, dst_t = parts[0], parts[-1]
                rel_t = '__'.join(parts[1:-1])

            ei = d[src_t, rel_t, dst_t].edge_index
            if ei.shape[1] == 0:
                continue

            x_in = (x[src_t], x[dst_t]) if src_t != dst_t else x[src_t]
            _, (_, alpha) = gat_conv(x_in, ei, return_attention_weights=True)
            # alpha: (E, H)
            alpha_cpu = alpha.cpu().float()
            n_edges, n_heads = alpha_cpu.shape
            dst_nodes = ei[1].cpu()

            # 노드별 집계 (head 평균 먼저)
            alpha_mean_head = alpha_cpu.mean(dim=1)  # (E,)
            unique_dst = dst_nodes.unique()
            n_dst = len(unique_dst)

            node_max_list, node_ent_list, node_gini_list = [], [], []
            for v in unique_dst:
                mask = (dst_nodes == v)
                a_v = alpha_mean_head[mask]  # 이 노드로 들어오는 edge들의 attention
                node_max_list.append(a_v.max().item())
                # entropy: -sum(p log p)
                a_v_safe = a_v.clamp(min=1e-9)
                ent = -(a_v_safe * a_v_safe.log()).sum().item()
                node_ent_list.append(ent)
                # gini coefficient
                a_sorted = a_v.sort().values
                n = len(a_sorted)
                if n > 1:
                    idx = torch.arange(1, n+1, dtype=torch.float32)
                    gini = (2 * (idx * a_sorted).sum() / (n * a_sorted.sum()) - (n+1)/n).item()
                else:
                    gini = 0.0
                node_gini_list.append(gini)

            layer_attn[(src_t, rel_t, dst_t)] = {
                'mean_alpha':   float(alpha_cpu.mean()),       # = N_dst/E (구조적)
                'node_max':     float(np.mean(node_max_list)), # 집중도
                'node_entropy': float(np.mean(node_ent_list)), # 선택성
                'node_gini':    float(np.mean(node_gini_list)),
                'n_edges': n_edges,
                'n_dst':   n_dst,
                'n_heads': n_heads,
            }

        layer_results.append(layer_attn)

        if layer_idx < 3:
            x_new = hconv(x, d.edge_index_dict)
            bn  = model.bns[layer_idx]
            act = _CONV_ACTS[layer_idx]
            x = {k: model.drop(act(bn[k](v.view(-1, model.hid)))) if k in bn
                 else model.drop(act(v)) for k, v in x_new.items()}

    return layer_results


print('extract_attention() 재정의 완료')
print('  mean_alpha  : N_dst/E (구조적, 두 모델에서 동일 — 참고용)')
print('  node_max    : 노드별 최대 attention 평균 (집중도)')
print('  node_entropy: 노드별 entropy 평균 (낮을수록 선택적)')
print('  node_gini   : 노드별 gini 평균 (높을수록 집중)')

In [ ]:
# Cell 8 — Attention 추출 실행

with torch.no_grad():
    attn_ipc   = extract_attention(model_ipc,   data_ipc_d,   has_ipc=True)
    attn_noipc = extract_attention(model_noipc, data_noipc_d, has_ipc=False)

print('추출 완료')
print(f'  phase3    : {len(attn_ipc)} layers, edge types: {list(attn_ipc[0].keys())}')
print(f'  phase3_noipc: {len(attn_noipc)} layers, edge types: {list(attn_noipc[0].keys())}')

In [ ]:
# Cell 9 — 레이어별 출력 (두 모델 나란히)

def print_layer_attn(layer_results, title):
    print(f'\n{"="*80}')
    print(f'  {title}')
    print(f'{"="*80}')
    for li, layer in enumerate(layer_results):
        if not layer: continue
        n_heads = list(layer.values())[0]['n_heads']
        print(f'\n  ── Layer {li+1}  (heads={n_heads}) {"─"*40}')
        print(f'  {"엣지 타입":<25} {"node_max":>9} {"entropy":>9} {"gini":>7}  {"n_dst":>6} / {"E":>6}')
        print(f'  {"-"*70}')
        for (src_t, rel_t, dst_t), s in sorted(layer.items(), key=lambda x: -x[1]['node_max']):
            label = f'{rel_t}  ({src_t[:2]}→{dst_t[:3]})'
            print(f'  {label:<25} {s["node_max"]:>9.4f} {s["node_entropy"]:>9.4f} {s["node_gini"]:>7.4f}  {s["n_dst"]:>6} / {s["n_edges"]:>6}')

print_layer_attn(attn_ipc,   'phase3  (IPC 포함)')
print_layer_attn(attn_noipc, 'phase3_noipc (IPC 제외)')

In [ ]:
# Cell 10 — 전체 레이어 평균 비교표 (node_max / entropy / gini)

def compute_overall(layer_results):
    et_vals = defaultdict(lambda: defaultdict(list))
    for layer in layer_results:
        for et, s in layer.items():
            for metric in ['node_max', 'node_entropy', 'node_gini', 'mean_alpha']:
                et_vals[et][metric].append(s[metric])
    return {et: {m: np.mean(v) for m, v in mv.items()} for et, mv in et_vals.items()}

overall_ipc   = compute_overall(attn_ipc)
overall_noipc = compute_overall(attn_noipc)

all_ets = sorted(set(overall_ipc.keys()) | set(overall_noipc.keys()),
                 key=lambda e: -overall_noipc.get(e, {}).get('node_max', -999))

W = 95
print(f'\n{"="*W}')
print(f'전체 4개 레이어 평균 — node_max (집중도, 높을수록 엣지 선택적 사용)')
print(f'{"─"*W}')
print(f'  {"엣지 타입":<26} {"phase3":>10} {"noipc":>10}  {"diff":>8}   '
      f'{"ent(ipc)":>10} {"ent(no)":>10}  {"diff":>8}')
print(f'{"─"*W}')
for et in all_ets:
    src_t, rel_t, dst_t = et
    ri = overall_ipc.get(et, {})
    rn = overall_noipc.get(et, {})
    mx_i = ri.get('node_max', float('nan'))
    mx_n = rn.get('node_max', float('nan'))
    en_i = ri.get('node_entropy', float('nan'))
    en_n = rn.get('node_entropy', float('nan'))
    label = f'{rel_t}  ({src_t[:2]}→{dst_t[:3]})'

    def fmt(v): return f'{v:.4f}' if not np.isnan(v) else '   N/A'
    def dfmt(a, b): return f'{b-a:+.4f}' if not (np.isnan(a) or np.isnan(b)) else '   N/A'

    print(f'  {label:<26} {fmt(mx_i):>10} {fmt(mx_n):>10}  {dfmt(mx_i,mx_n):>8}   '
          f'{fmt(en_i):>10} {fmt(en_n):>10}  {dfmt(en_i,en_n):>8}')
print(f'{"="*W}')
print('  node_max : 각 dst 노드의 최대 attention 평균 (높을수록 특정 엣지에 집중)')
print('  entropy  : 각 dst 노드의 attention entropy 평균 (낮을수록 선택적)')

In [ ]:
# Cell 11 — 레이어별 변화 추이 시각화 (node_max 기준)

COMMON_ETS = [
    ('patent',  'cites',    'patent'),
    ('patent',  'cited_by', 'patent'),
    ('company', 'owns',     'patent'),
    ('patent',  'owned_by', 'company'),
    ('patent',  'sued',     'patent'),
    ('patent',  'sued_by',  'patent'),
]
IPC_ONLY_ETS = [
    ('patent', 'has_ipc',     'ipc'),
    ('ipc',    'rev_has_ipc', 'patent'),
]

COLORS = {
    'cites':       '#1976D2',
    'cited_by':    '#42A5F5',
    'owns':        '#388E3C',
    'owned_by':    '#81C784',
    'sued':        '#E53935',
    'sued_by':     '#EF9A9A',
    'has_ipc':     '#F57C00',
    'rev_has_ipc': '#FFB74D',
}
MARKERS = ['o','s','D','^','v','P','*','X']

layers = [f'L{i+1}' for i in range(4)]

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=False)

def plot_model(ax, layer_results, title, ets_to_plot):
    for i, et in enumerate(ets_to_plot):
        src_t, rel_t, dst_t = et
        vals = [layer[et]['node_max'] if et in layer else np.nan
                for layer in layer_results]
        color  = COLORS.get(rel_t, f'C{i}')
        marker = MARKERS[i % len(MARKERS)]
        label  = f'{rel_t}  ({src_t[:2]}→{dst_t[:3]})'
        ax.plot(layers, vals, marker=marker, color=color, linewidth=1.8,
                markersize=7, label=label)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Layer')
    ax.set_ylabel('node_max (집중도)')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)

plot_model(axes[0], attn_ipc,   'phase3  (IPC 포함)',      COMMON_ETS + IPC_ONLY_ETS)
plot_model(axes[1], attn_noipc, 'phase3_noipc (IPC 제외)', COMMON_ETS)

plt.suptitle('GAT node_max per Edge Type per Layer', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('gat_attention_layer_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: gat_attention_layer_trend.png')

In [ ]:
# Cell 12 — 전체 평균 막대 그래프 비교 (node_max 기준)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

def bar_model(ax, overall, title):
    sorted_ets = sorted(overall.items(), key=lambda x: -x[1]['node_max'])
    labels = [f'{e[1]}\n({e[0][:2]}→{e[2][:3]})' for (e, _) in sorted_ets]
    vals   = [v['node_max'] for _, v in sorted_ets]
    colors = [COLORS.get(e[1], '#999') for e, _ in sorted_ets]
    bars   = ax.bar(range(len(vals)), vals, color=colors, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.4f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel('node_max (4-layer avg)')
    ax.set_title(title, fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')

bar_model(axes[0], overall_ipc,   'phase3  (IPC 포함)')
bar_model(axes[1], overall_noipc, 'phase3_noipc (IPC 제외)')

plt.suptitle('전체 레이어 평균 node_max — 엣지 타입별', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('gat_attention_overall_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: gat_attention_overall_bar.png')

In [ ]:
# Cell 13 — 공통 엣지 타입 산점도 비교 (node_max 기준)

common_ets = [et for et in COMMON_ETS if et in overall_ipc and et in overall_noipc]

fig, ax = plt.subplots(figsize=(7, 6))

for i, et in enumerate(common_ets):
    src_t, rel_t, dst_t = et
    x = overall_ipc[et]['node_max']
    y = overall_noipc[et]['node_max']
    color  = COLORS.get(rel_t, f'C{i}')
    marker = MARKERS[i % len(MARKERS)]
    ax.scatter(x, y, c=color, marker=marker, s=100, zorder=3,
               label=f'{rel_t}  ({src_t[:2]}→{dst_t[:3]})')
    ax.annotate(rel_t, (x, y), textcoords='offset points', xytext=(6, 4), fontsize=8)

lim_min = min(ax.get_xlim()[0], ax.get_ylim()[0])
lim_max = max(ax.get_xlim()[1], ax.get_ylim()[1])
ax.plot([lim_min, lim_max], [lim_min, lim_max], 'k--', linewidth=0.8, alpha=0.4)

ax.set_xlabel('phase3  (IPC 포함)  — node_max')
ax.set_ylabel('phase3_noipc  — node_max')
ax.set_title('공통 엣지 타입 node_max 비교\n(대각선 위 = noipc에서 더 집중)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('gat_attention_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: gat_attention_scatter.png')

# Edge-Type Ablation — gnn_266_phase3 (IPC 포함 모델)

각 엣지 타입을 하나씩 제거한 뒤 patent 임베딩 변화량(cosine distance)과
test AUROC / MRR / H@10 변화를 측정해 엣지별 기여도를 비교한다.

> **측정 기준**
> - `Δcosine` : `1 - cosine_sim(h_full, h_ablated)` 평균  → 클수록 해당 엣지가 임베딩에 영향이 큼
> - `ΔAUROC`  : full 모델 AUROC - ablated AUROC              → 클수록 task에 중요
> - `ΔMRR`    : full 모델 MRR   - ablated MRR

In [ ]:
# Cell A — test_edges 재구성 + 평가 함수

from sklearn.metrics import roc_auc_score, average_precision_score

test_cases  = set(ckpt_ipc['test_cases'])
val_cases   = set(ckpt_ipc['val_cases'])

te_mask = np.array([c in test_cases for c in sued_case])
va_mask = np.array([c in val_cases  for c in sued_case])
test_edges = sued_edges[:, te_mask]
val_edges  = sued_edges[:, va_mask]

print(f'Val  edges: {val_edges.shape[1]}')
print(f'Test edges: {test_edges.shape[1]}')


@torch.no_grad()
def evaluate_h(h_pat, edges, seed=99):
    """임베딩 h에서 AUROC / AUPRC / MRR / H@k 계산."""
    edges_cpu = edges.cpu()   # GPU tensor 대비
    h = F.normalize(h_pat, dim=1)
    dev = h.device
    pos = (h[edges_cpu[0].to(dev)] * h[edges_cpu[1].to(dev)]).sum(1).cpu().numpy()
    rng   = np.random.default_rng(seed)
    neg_d = torch.tensor(rng.integers(0, len(pat_list), len(pos)), dtype=torch.long).to(dev)
    neg   = (h[edges_cpu[0].to(dev)] * h[neg_d]).sum(1).cpu().numpy()

    scores = np.concatenate([pos, neg])
    labels = np.array([1]*len(pos) + [0]*len(neg))
    auroc  = roc_auc_score(labels, scores)
    auprc  = average_precision_score(labels, scores)

    # Ranking
    h_np = h.cpu().numpy()
    plt_to_defs = defaultdict(set)
    for s, d in zip(edges_cpu[0].numpy(), edges_cpu[1].numpy()):
        plt_to_defs[s].add(d)
    ranks = []
    for p, ds in plt_to_defs.items():
        sims = h_np @ h_np[p]; sims[p] = -np.inf
        order = np.argsort(-sims)
        ro = {int(x): r+1 for r, x in enumerate(order)}
        for d in ds: ranks.append(ro[d])
    ranks = np.array(ranks)
    return {
        'auroc': auroc, 'auprc': auprc,
        'mrr':   float(np.mean(1/ranks)),
        'h10':   float(np.mean(ranks <= 10)),
        'med':   float(np.median(ranks)),
    }

print('evaluate_h() 정의 완료')

In [ ]:
# Cell B — forward_ablated: 특정 엣지 타입을 제거하고 forward

@torch.no_grad()
def forward_ablated(model, hetero_data, skip_ets):
    """
    skip_ets: set of edge type tuples to remove, e.g. {('patent','has_ipc','ipc')}
    노드가 incoming edge를 잃으면 이전 레이어 표현을 유지 (HeteroConv 기본 동작).
    """
    model.eval()
    d = hetero_data

    # skip_ets를 제거한 edge_index_dict
    ei_dict = {et: ei for et, ei in d.edge_index_dict.items()
               if et not in skip_ets}

    x_pat = torch.cat([d['patent'].struct, d['patent'].x], dim=1)
    x = {
        'patent':  model.pat_proj(x_pat),
        'company': F.leaky_relu(model.co_proj(d['company'].x)),
        'ipc':     model.ipc_emb.weight,
    }

    for conv, bn, act in zip(model.convs[:3], model.bns, _CONV_ACTS):
        x_new = conv(x, ei_dict)
        # 메시지를 받지 못한 노드 타입은 이전 표현 유지
        for k in x:
            if k not in x_new:
                x_new[k] = x[k]
        x = {k: model.drop(act(bn[k](v.view(-1, model.hid)))) if k in bn
             else model.drop(act(v)) for k, v in x_new.items()}

    x_out = model.convs[3](x, ei_dict)
    for k in x:
        if k not in x_out:
            x_out[k] = x[k]
    return {k: v.view(-1, model.out) if v.shape[-1] != model.out else v
            for k, v in x_out.items()}


print('forward_ablated() 정의 완료')

In [ ]:
# Cell C — Edge-Type Ablation (gnn_266_phase3)

te = test_edges.to(device)

# Baseline: full model
with torch.no_grad():
    h_full = model_ipc(data_ipc_d)['patent']
    full_result = evaluate_h(h_full, te)

print(f'[Full model]  AUROC={full_result["auroc"]:.4f}  MRR={full_result["mrr"]:.4f}  '
      f'H@10={full_result["h10"]:.4f}  Med={full_result["med"]:.0f}')
print()

ABLATION_TARGETS = [
    ('cites',           {('patent', 'cites',       'patent')}),
    ('cited_by',        {('patent', 'cited_by',    'patent')}),
    ('has_ipc',         {('patent', 'has_ipc',     'ipc'   )}),
    ('rev_has_ipc',     {('ipc',    'rev_has_ipc', 'patent')}),
    ('IPC_all',         {('patent', 'has_ipc',     'ipc'),
                         ('ipc',    'rev_has_ipc', 'patent')}),
    ('owns',            {('company','owns',         'patent')}),
    ('owned_by',        {('patent', 'owned_by',    'company')}),
    ('sued',            {('patent', 'sued',        'patent')}),
    ('sued_by',         {('patent', 'sued_by',     'patent')}),
    ('sued_all',        {('patent', 'sued',        'patent'),
                         ('patent', 'sued_by',     'patent')}),
]

results = []
for label, skip_ets in ABLATION_TARGETS:
    with torch.no_grad():
        h_ab = forward_ablated(model_ipc, data_ipc_d, skip_ets)['patent']
        r    = evaluate_h(h_ab, te)

    h_f = F.normalize(h_full, dim=1).cpu()
    h_a = F.normalize(h_ab,   dim=1).cpu()
    delta_cos = (1 - (h_f * h_a).sum(dim=1)).mean().item()

    results.append({
        'edge':        label,
        'delta_cos':   delta_cos,
        'delta_auroc': full_result['auroc'] - r['auroc'],
        'delta_mrr':   full_result['mrr']   - r['mrr'],
        'delta_h10':   full_result['h10']   - r['h10'],
        'delta_med':   r['med'] - full_result['med'],
        'auroc': r['auroc'], 'mrr': r['mrr'],
        'h10':   r['h10'],   'med': r['med'],
    })

# Print table
W = 82
print(f'{"edge":<14} {"d_cos":>7} {"d_AUROC":>9} {"d_MRR":>8} {"d_H@10":>8} {"d_Med":>7}  '
      f'{"AUROC":>7} {"MRR":>7} {"H@10":>7} {"Med":>5}')
print('-' * W)
for r in results:
    print(f'{r["edge"]:<14} {r["delta_cos"]:>7.4f} {r["delta_auroc"]:>+9.4f} '
          f'{r["delta_mrr"]:>+8.4f} {r["delta_h10"]:>+8.4f} {r["delta_med"]:>+7.0f}  '
          f'{r["auroc"]:>7.4f} {r["mrr"]:>7.4f} {r["h10"]:>7.4f} {r["med"]:>5.0f}')
print('-' * W)
print('(positive delta = worse than full model => that edge matters)')

In [ ]:
# Cell D — Visualization

from matplotlib.patches import Patch

df_ab = pd.DataFrame(results)

IPC_EDGES = {'has_ipc', 'rev_has_ipc', 'IPC_all'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = [
    ('delta_cos',   'delta Cosine Distance\n(embedding shift, larger = more impact)'),
    ('delta_auroc', 'delta AUROC\n(perf drop, larger = more important)'),
    ('delta_mrr',   'delta MRR\n(ranking drop, larger = more important)'),
]

for ax, (col, title) in zip(axes, metrics):
    vals   = df_ab[col].values
    labels = df_ab['edge'].values
    colors = ['#E53935' if l in IPC_EDGES else '#1976D2' for l in labels]
    bars   = ax.barh(range(len(vals)), vals, color=colors, alpha=0.85)
    ax.set_yticks(range(len(vals)))
    ax.set_yticklabels(labels, fontsize=9)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(title, fontsize=10)
    ax.grid(True, alpha=0.3, axis='x')
    vmax = max(abs(vals))
    for bar, v in zip(bars, vals):
        ax.text(max(v, 0) + vmax * 0.01, bar.get_y() + bar.get_height() / 2,
                f'{v:+.4f}', va='center', fontsize=8)

axes[0].legend(handles=[Patch(color='#E53935', label='IPC edges'),
                         Patch(color='#1976D2', label='Other edges')],
               fontsize=8, loc='lower right')

plt.suptitle('Edge-Type Ablation — gnn_266_phase3\n'
             '(positive delta = worse without that edge => it contributes)',
             fontsize=12)
plt.tight_layout()
plt.savefig('ablation_phase3.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ablation_phase3.png')